# 15 · Original selected pipeline: short version

[Guide](../README.md) | [Previous](14_Interpretation_and_Error_Analysis.ipynb) | [Next](16_Complete_ML_Pipeline.ipynb)

## 1. What problem are we solving?

How do we turn the lesson choices into one short, runnable pipeline?

## 2. Why does it matter?

The same preparation must be applied consistently to training and future inputs.

## 3. Key concepts

| Term | Simple meaning |
|---|---|
| Fixed recipe | Previously selected preparation steps and model settings; no new search. |
| Fit / predict | Learn from labeled training rows / apply the learned pipeline to new inputs. |
| Refit | Fit the same settings again using all labeled rows after making method choices. |
| Submission | One ID and one predicted goal/miss label for every competition-test row. |
| Pipeline notation | `clone` copies unfitted settings; `step__setting` changes one pipeline setting. Earlier steps stay fixed during a comparison. |

## 4. Intuitive example

For a new kick with blank capacity, fill using the training median, encode its categories, and add input-only features such as `70 − 30 = 40` for the form gap. If the fitted pipeline returns 0.60, cutoff 0.50 produces goal=1. The filling value is learned during fitting, never recomputed from the test file.

## 5. Fixed pipeline and output

**Original recipe:** drop incomplete rows while fitting; use cutoff **0.50**. Refit this fixed recipe on labeled training data, predict every test row, and export the two-column file. This notebook does not repeat the search.

Notebook 16 is a **revised recipe**, not an identical copy: it keeps/imputes incomplete rows and selects its cutoff using training-only predictions.

## 6. What we retained from previous notebooks

| Phase | Chosen in the lesson |
|---|---|
| Missing · 03 | Drop incomplete training rows |
| Scaling · 03 | Robust scaling |
| Encoding · 04 | One-hot |
| Features · 05 | All candidate groups |
| Selection · 06 | Drop suspected decoys + L1 |
| Reduction · 07 | No PCA |
| Outliers · 08 | Add a flag |
| Balancing · 09 | Original class counts |
| Model · 10 | CatBoost |
| Tuning · 11 | Random search |
| Ensemble · 12 | Single model |
| Probabilities · 13 | Original probabilities |
| Decision cutoff · 13 | 0.50 |

This is the original lesson-selected recipe: row dropping during fitting and cutoff 0.50. Notebook 16 documents the later follow-up recipe.

## 7. Setup

**Reusable setup — run this cell, then skip to the lesson.** You do not need to study all these lines now: they load the data and rebuild earlier steps so this notebook works on its own. Revisit the earlier lessons for explanations.

Change `DATA_DIR` for your runtime. The cell starts collapsed where the editor supports it; GitHub may show it expanded. `X` contains inputs; `y` contains known answers. Learned preparation stays inside training folds.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.precision", 3)

# Change this one folder on Kaggle or Colab (see the guide).
DATA_DIR = "../data"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Train:", train.shape, "Test:", test.shape)

# Split train.csv: 80% for training, 20% for a separate validation check.
training_data, validation_data = train_test_split(
    train, test_size=0.20, stratify=train["is_scored"], random_state=2026
)
training_data = training_data.reset_index(drop=True)

# IDs identify rows; they do not describe how a penalty was taken.
id_columns = ["id", "kicker_id", "keeper_id", "match_id"]
X = training_data.drop(columns=id_columns + ["is_scored"])
y = training_data["is_scored"]

from sklearn.base import clone
from sklearn.compose import make_column_selector
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.feature_selection import SelectFromModel
from imblearn import FunctionSampler
from imblearn.pipeline import Pipeline
from catboost import CatBoostClassifier

def remove_incomplete_rows(X, y):
    keep = ~X.isna().any(axis=1)
    return X.loc[keep], y[keep]

def drop_suspected_decoys(data):
    return data.drop(columns=["jersey_color", "referee_strictness"], errors="ignore")

def add_relative_features(data):
    data = data.copy()
    data["skill_gap"] = data["player_last5_scored_percent"] - data["goalkeeper_last5_saved_percent"]
    data["foot_advantage"] = data["strong_foot_usage"] - data["weak_foot_usage"]
    data["form_ratio"] = (data["player_last5_scored_percent"] + 1) / (data["goalkeeper_last5_saved_percent"] + 1)
    data["form_interaction"] = data["player_last5_scored_percent"] * data["goalkeeper_last5_saved_percent"] / 100
    data["player_form_squared"] = (data["player_last5_scored_percent"] / 100) ** 2
    data["keeper_form_squared"] = (data["goalkeeper_last5_saved_percent"] / 100) ** 2
    return data

def add_pressure_features(data):
    data = data.copy()
    data["late_close_game"] = ((data["minute_taken"] >= 75) & (data["current_score_diff"].abs() <= 1)).astype(int)
    gap = data["player_last5_scored_percent"] - data["goalkeeper_last5_saved_percent"]
    data["skill_under_pressure"] = gap * data["late_close_game"]
    data["after_90_minutes"] = (data["minute_taken"] > 90).astype(int)
    data["trailing"] = (data["current_score_diff"] < 0).astype(int)
    data["tied_game"] = (data["current_score_diff"] == 0).astype(int)
    data["minute_score_interaction"] = data["minute_taken"] * data["current_score_diff"]
    data["crowd_support"] = (data["home_or_away"] == data["crowd_behind_goal"]).astype(int)
    data["late_final"] = ((data["minute_taken"] >= 75) & (data["match_type"] == "Final")).astype(int)
    return data

def add_environment_features(data):
    data = data.copy()
    data["poor_pitch"] = (data["pitch_condition"] == "Poor").astype(int)
    data["poor_pitch_late"] = data["poor_pitch"] * (data["minute_taken"] >= 75)
    data["humidity_weak_foot"] = data["weather_humidity"] / 100 * data["weak_foot_usage"]
    data["humidity_squared"] = (data["weather_humidity"] / 100) ** 2
    data["temperature_squared"] = data["temperature"] ** 2
    data["log_capacity"] = np.log1p(data["stadium_capacity"])
    return data

def add_features(data):
    data = add_relative_features(data)
    data = add_pressure_features(data)
    return add_environment_features(data)

def suspicious_rows(data):
    # Screening rules, not proof: unusual time, cold weather, or very large venue.
    return ((data["minute_taken"] > 120) |
            (data["temperature"] < 0) |
            (data["stadium_capacity"] > 110000))

def add_outlier_flag(data):
    data = data.copy()
    data["unusual_kick"] = suspicious_rows(data).astype(int)
    return data

numbers = Pipeline([("fill", SimpleImputer(strategy='median')), ("scale", RobustScaler())])
categories = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("numbers", numbers, make_column_selector(dtype_include="number")),
    ("categories", categories, make_column_selector(dtype_exclude="number")),
])

pipeline = Pipeline([
    ("missing_rows", FunctionSampler(func=remove_incomplete_rows, validate=False)),
    ("outlier_values", FunctionTransformer(add_outlier_flag)),
    ("features", FunctionTransformer(add_features)),
    ("decoys", FunctionTransformer(drop_suspected_decoys)),
    ("prepare", preprocess),
    ("selection", SelectFromModel(LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, random_state=42))),
    ("model", CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1, loss_function='Logloss', random_seed=42, thread_count=2, verbose=False, allow_writing_files=False)),
])

selected_model = clone(pipeline)

Train: (20000, 20) Test: (6000, 19)


## 8. Experiments

Run the topic-specific steps below. Saved experimental outputs are retained from the earlier runs; regrouping setup does not create a new evaluation.

### Train the unchanged recipe using the full training file
The same training-row cleaning rule still applies. Every competition-test row gets a prediction.

In [17]:
final_model = clone(selected_model)
X_all = train.drop(columns=id_columns + ["is_scored"])
y_all = train["is_scored"]
X_test = test.drop(columns=id_columns)
final_model.fit(X_all, y_all)
prediction = final_model.predict(X_test)
submission = pd.DataFrame({"id": test["id"], "is_scored": prediction})
assert len(submission) == len(test)
assert submission["id"].is_unique and submission["is_scored"].isin([0, 1]).all()
display(submission.head())

,id,is_scored
0,20000,1
1,20001,0
2,20002,1
3,20003,0
4,20004,1


In [18]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
submission.to_csv("../results/15_simple_submission.csv", index=False)
print("Saved the competition predictions.")

Saved the competition predictions.


## 9. Results and decision

| Question | Answer |
|---|---|
| What did we compare? | No new search: apply the original selected recipe and export one prediction per test row. |
| Which performed best? | No new comparison or F1 calculation here; apply the previously selected CatBoost recipe. |
| Why did we choose it? | One fixed pipeline keeps training and prediction-time processing consistent. |
| What are its limitations? | This short notebook does not compute a new validation F1. Its original row-dropping choice differs from the follow-up recipe in Notebook 16. |
| What continues to the next notebook? | Export the original fixed recipe; Notebook 16 presents the follow-up complete pipeline. |

## 10. Key takeaways and next step

- Refit the selected recipe rather than rerun all method searches.
- Use the same learned preparation for every future input.

Next: [16 Complete ML Pipeline](16_Complete_ML_Pipeline.ipynb).